# Digital Image Enhancement — Spatial Filtering Module

This notebook demonstrates the **5 essential spatial filtering techniques** in Digital Image Processing:
1. **Mean Filter (Box Filter)** — Linear Smoothing
2. **Gaussian Filter** — Weighted Linear Smoothing
3. **Median Filter** — Non-Linear Order-Statistic Smoothing
4. **Sobel Filter** — 1st Derivative Edge Detection
5. **Laplacian Filter** — 2nd Derivative Image Sharpening

We test these filters on `original_image.png` with simulated **Gaussian Noise** and **Salt-and-Pepper Noise** to evaluate both visual quality and quantitative metrics (PSNR, MSE, Entropy, Contrast).

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Import custom modular functions from spatial-filtering package
from noise import add_gaussian_noise, add_salt_and_pepper_noise
from filters import (
    apply_mean_filter_cv, apply_gaussian_filter_cv, apply_median_filter_cv,
    apply_sobel_filter, apply_laplacian_filter
)
from metrics import calculate_stats, calculate_mse, calculate_psnr

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## Step 1: Load Original Reference Image
We load `original_image.png` in grayscale mode.

In [ ]:
img_path = '../original_image.png'
if not os.path.exists(img_path):
    img_path = 'original_image.png'

orig_bgr = cv2.imread(img_path)
orig_gray = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2GRAY)

plt.figure(figsize=(6, 8))
plt.imshow(orig_gray, cmap='gray')
plt.title(f'Original Reference ({orig_gray.shape[1]}x{orig_gray.shape[0]})', fontsize=12, fontweight='bold')
plt.axis('off')
plt.show()

stats = calculate_stats(orig_gray)
print(f"Original Stats -> Mean: {stats['mean']}, Std (Contrast): {stats['std']}, Entropy: {stats['entropy']} bits")

## Step 2: Smoothing Filters (Mean vs. Gaussian Blur)
We inject **Gaussian additive noise** ($σ = 25$) and evaluate **Mean (Box)** vs. **Gaussian Filter**.

In [ ]:
gaussian_noisy = add_gaussian_noise(orig_gray, mean=0, sigma=25.0)
mean_3x3 = apply_mean_filter_cv(gaussian_noisy, kernel_size=3)
mean_5x5 = apply_mean_filter_cv(gaussian_noisy, kernel_size=5)
gauss_5x5 = apply_gaussian_filter_cv(gaussian_noisy, kernel_size=5, sigma=1.5)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].imshow(gaussian_noisy, cmap='gray')
axes[0, 0].set_title('Gaussian Noisy Input (σ=25)', fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(mean_3x3, cmap='gray')
axes[0, 1].set_title('Mean Filter (3x3 Box)', fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(mean_5x5, cmap='gray')
axes[1, 0].set_title('Mean Filter (5x5 Box)', fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(gauss_5x5, cmap='gray')
axes[1, 1].set_title('Gaussian Filter (5x5, σ=1.5)', fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## Step 3: Median Filter (Removing Salt-and-Pepper Noise)
Salt-and-Pepper noise introduces extreme outliers (0 or 255). A **Median Filter** removes these dots while preserving sharp edges.

In [ ]:
sp_noisy = add_salt_and_pepper_noise(orig_gray, salt_prob=0.03, pepper_prob=0.03)
mean_sp = apply_mean_filter_cv(sp_noisy, kernel_size=3)
median_sp = apply_median_filter_cv(sp_noisy, kernel_size=3)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(sp_noisy, cmap='gray')
axes[0].set_title('Salt & Pepper Noise (6% Density)', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(mean_sp, cmap='gray')
axes[1].set_title(f'Mean Filter (3x3) - PSNR: {calculate_psnr(orig_gray, mean_sp)} dB', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(median_sp, cmap='gray')
axes[2].set_title(f'Median Filter (3x3) - PSNR: {calculate_psnr(orig_gray, median_sp)} dB', fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## Step 4: Sobel Filter (1st Derivative Edge Detection)
Computes horizontal gradient $G_x$, vertical gradient $G_y$, gradient magnitude $M(x,y)$, and angle $θ(x,y)$.

In [ ]:
Gx, Gy, mag_sobel, angle_sobel = apply_sobel_filter(orig_gray, ksize=3)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].imshow(Gx, cmap='gray')
axes[0, 0].set_title('Sobel Gx (Horizontal Derivative)', fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(Gy, cmap='gray')
axes[0, 1].set_title('Sobel Gy (Vertical Derivative)', fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(mag_sobel, cmap='gray')
axes[1, 0].set_title('Sobel Magnitude M = sqrt(Gx² + Gy²)', fontweight='bold')
axes[1, 0].axis('off')

im = axes[1, 1].imshow(angle_sobel, cmap='twilight')
axes[1, 1].set_title('Sobel Direction Angle θ', fontweight='bold')
axes[1, 1].axis('off')
fig.colorbar(im, ax=axes[1, 1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## Step 5: Laplacian Filter (2nd Derivative Sharpening)
Computes 2nd spatial derivatives $\nabla^2 f$ and sharpens the original image.

In [ ]:
laplacian_map, laplacian_sharpened = apply_laplacian_filter(orig_gray, ksize=3, alpha=0.7)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(orig_gray, cmap='gray')
axes[0].set_title('Original Grayscale Reference', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(laplacian_map, cmap='gray')
axes[1].set_title('Laplacian Response |∇²f|', fontweight='bold')
axes[1].axis('off')

axes[2].imshow(laplacian_sharpened, cmap='gray')
axes[2].set_title('Sharpened Image: g = f - α∇²f', fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()